# Your first DataFrame: read, transform and write with PySpark

A PySpark DataFrame is a distributed table with named columns. You create or read one, chain transformations such as select, filter, withColumn and groupBy, and nothing runs until an action such as display, count or a write asks for results. Every SQL clause you learned has a DataFrame method.

**How to run:** attach **Serverless** compute (top right), then run the cells from top to bottom. Each part asks you to predict the result first; the expected results are listed at the end.

From the Learning Hub lesson: [Your first DataFrame: read, transform and write with PySpark](https://www.datapreplab.com/#topic/pyspark-first-dataframe) · Independent study resource, not affiliated with or endorsed by Databricks.

**Goal:** build, transform, inspect and save a DataFrame, and see laziness in action.

**Part A: create.**

In [ ]:
from pyspark.sql import functions as F

orders = spark.createDataFrame(
    [(1, "UK", 120), (2, "FR", 80), (3, "UK", 50), (4, "DE", 90), (5, "FR", -10)],
    ["order_id", "region", "amount"])
display(orders)

**Part B: laziness.** Run this and notice it returns at once with no rows:

In [ ]:
step = orders.filter(F.col("amount") > 100).select("order_id")
step

Then display it: now Spark computes it.

In [ ]:
display(step)

**Part C: the full chain.** Predict which region comes first, then run:

In [ ]:
revenue = (orders
    .filter(F.col("amount") > 0)
    .withColumn("amount_with_vat", F.col("amount") * 1.2)
    .groupBy("region")
    .agg(F.sum("amount").alias("revenue"),
         F.round(F.sum("amount_with_vat"), 2).alias("revenue_with_vat"))
    .orderBy(F.desc("revenue")))

display(revenue)
revenue.write.mode("overwrite").saveAsTable("workspace.default.revenue_by_region")

**Part D: same answer in SQL.**

In [ ]:
orders.createOrReplaceTempView("orders_v")
display(spark.sql("""
  SELECT region, sum(amount) AS revenue FROM orders_v
  WHERE amount > 0 GROUP BY region ORDER BY revenue DESC"""))

**Part E: read it back.**

In [ ]:
display(spark.table("workspace.default.revenue_by_region"))

Expected evidence:

```text
Part A -> 5 rows
Part B -> a DataFrame description, then order 1 only
Part C -> UK 170 204.0 ; DE 90 108.0 ; FR 80 96.0
Part D -> UK 170 ; DE 90 ; FR 80
Part E -> the same three rows from the saved table
```

**Cleanup:**

In [ ]:
spark.sql("DROP TABLE IF EXISTS workspace.default.revenue_by_region")